In [97]:
import pandas as pd
import glob
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import random

In [112]:
NUM_MATCHES = 1
MATCH_DIST = 10
ref_length_min = 8 
ref_length_max = 12  # Maximum length of reference sequence in seconds
query_length_min =0.4
query_length_max = 1
exp_no = "4"
folder_q = r"C:\Arjun\Thesis\data\20200422_172431-sunset2\unormal_subs"
gps_query_path = r"C:\Arjun\Thesis\data\20200422_172431-sunset2\sunset2_gps_v2.csv"
# AFTER
folder_r_list = [
    r"C:\Arjun\Thesis\data\20200421_170039-sunset1\filtered chunks\un-normalised",  #"sunset1"
    r"C:\Arjun\Thesis\data\Results\night_bp45_thr005_tau30_nonorm",  #night
    r"C:\Arjun\Thesis\data\Results\daytime_bp45_thr005_tau30_nonorm",  #daytime
    r"C:\Arjun\Thesis\data\Results\morning_bp45_thr005_tau30_nonorm",  #morning
]
#ref 1 s1
#ref 2 night
#ref 3 daytime
#ref 4 morning
ref_gps_paths = [
    r"C:\Arjun\Thesis\data\20200421_170039-sunset1\sunset1_gps_v2.csv", #s1
    r"C:\Arjun\Thesis\data\20200427_181204-night\night_gps_v2.csv", #night,
    r"C:\Arjun\Thesis\data\20200424_daytime\daytime_gps_v2.csv",  
    r"C:\Arjun\Thesis\data\2020-04-28-09-14-11-morning\morning_gps_v2.csv"
]
exp_path = r"C:\Arjun\Thesis\results\exp_4_all"

In [111]:

def unix_to_brisbane(unix_time):
    """Convert UNIX timestamp to Brisbane time (UTC+10)"""
    # Brisbane is UTC+10 (no daylight saving)
    brisbane_time = datetime.fromtimestamp(unix_time, tz=timezone.utc) + pd.Timedelta(hours=10)
    return brisbane_time.strftime('%Y-%m-%d %H:%M:%S')

In [113]:
def get_timestamps(folder_path):
    """Extract 4th column timestamps from all CSVs in folder"""
    timestamps = []
    for csv_file in glob.glob(f"{folder_path}/*.csv"):
        df = pd.read_csv(csv_file)
        ts_col = df.iloc[:, 3]  # 4th column (0-indexed)
        timestamps.extend(ts_col.dropna().tolist())
    return set(timestamps)

In [114]:
ts_r_list = [get_timestamps(f) for f in folder_r_list]
ts_q = get_timestamps(folder_q)



In [102]:
min_r_list = []
max_r_list = []
for i, ts_r in enumerate(ts_r_list):
    min_r = unix_to_brisbane(min(ts_r))
    max_r = unix_to_brisbane(max(ts_r))
    min_r_list.append(min_r)
    max_r_list.append(max_r)
    print(f"Ref {i+1} - Min: {min_r}, Max: {max_r}, {int((max(ts_r)-min(ts_r))/60)} min {int((max(ts_r)-min(ts_r))%60)} sec")

min_q = unix_to_brisbane(min(ts_q))
max_q = unix_to_brisbane(max(ts_q))

query_min_ts = min(ts_q)
query_max_ts = max(ts_q)

print(f"Query - Min: {min_q}, Max: {max_q}")
print(f"Query {int((max(ts_q)-min(ts_q))/60)} min {int((max(ts_q)-min(ts_q))%60)} sec")

Ref 1 - Min: 2020-04-21 17:03:04, Max: 2020-04-21 17:07:08, 4 min 3 sec
Ref 2 - Min: 2020-04-27 18:13:30, Max: 2020-04-27 18:16:44, 3 min 13 sec
Ref 3 - Min: 2020-04-24 15:12:03, Max: 2020-04-24 15:15:41, 3 min 37 sec
Ref 4 - Min: 2020-04-28 09:14:11, Max: 2020-04-28 09:17:26, 3 min 15 sec
Query - Min: 2020-04-22 17:24:21, Max: 2020-04-22 17:28:10
Query 3 min 49 sec


In [103]:
ref_gps_list = [pd.read_csv(p) for p in ref_gps_paths]
query_gps = pd.read_csv(gps_query_path)

In [115]:
def haversine_distance(lat1, lon1, lat2, lon2):
    # Earth radius in meters
    R = 6371000

    # Convert latitude and longitude from degrees to radians
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)

    # Calculate differences between latitudes and longitudes
    d_lat = lat2_rad - lat1_rad
    d_lon = lon2_rad - lon1_rad

    # Haversine formula
    a = np.sin(d_lat / 2) ** 2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(d_lon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    # Calculate the distance
    distance = R * c

    return distance

In [116]:
# AFTER
ref_gps_list_filtered = []
for i, (ref_gps, ts_r) in enumerate(zip(ref_gps_list, ts_r_list)):
    ref_min_ts = min(ts_r)
    ref_max_ts = max(ts_r)
    print(i)
    print(f"min, {unix_to_brisbane(ref_min_ts)}")
    print(f"max, {unix_to_brisbane(ref_max_ts)}")
    filtered = ref_gps[
        (ref_gps["elapsed_time_ts"] >= ref_min_ts) &
        (ref_gps["elapsed_time_ts"] <= ref_max_ts)
    ].reset_index(drop=True)
    ref_gps_list_filtered.append(filtered)
    print(f"ref_gps[{i+1}] truncated: {len(filtered)} rows")

# query_gps filtering stays exactly the same
query_gps = query_gps[
    (query_gps["elapsed_time_ts"] >= query_min_ts) &
    (query_gps["elapsed_time_ts"] <= query_max_ts)
].reset_index(drop=True)
print(f"query_gps[{i+1}] truncated: {len(query_gps)} rows")

0
min, 2020-04-21 17:03:04
max, 2020-04-21 17:07:08
ref_gps[1] truncated: 220 rows
1
min, 2020-04-27 18:13:30
max, 2020-04-27 18:16:44
ref_gps[2] truncated: 15 rows
2
min, 2020-04-24 15:12:03
max, 2020-04-24 15:15:41
ref_gps[3] truncated: 18 rows
3
min, 2020-04-28 09:14:11
max, 2020-04-28 09:17:26
ref_gps[4] truncated: 17 rows
query_gps[4] truncated: 201 rows


In [117]:
ref_lat_list = [g['latitude']         for g in ref_gps_list_filtered]
ref_lon_list = [g['longitude']        for g in ref_gps_list_filtered]
ref_ts_list  = [g['elapsed_time_ts']  for g in ref_gps_list_filtered]

query_lat = query_gps['latitude']
query_lon = query_gps['longitude']
query_ts = query_gps['elapsed_time_ts']

In [ ]:
def find_nearest_gps_match(ref_lat, ref_lon, ref_ts, query_lat, query_lon, query_ts, window_r, window_q, match_dist):
    """Find the nearest GPS match in query for a single reference point (only if within match_dist meters)"""
    best_match = None   
    for j, (qlat, qlon, qts) in enumerate(zip(query_lat, query_lon, query_ts)):
        dist = haversine_distance(ref_lat, ref_lon, qlat, qlon)
        
        if dist <= match_dist:
            best_match = {
                'ref_lat': ref_lat, 
                'ref_lon': ref_lon,
                'query_lat': qlat, 
                'query_lon': qlon,
                'ref_ts': ref_ts,
                'ref_time': unix_to_brisbane(ref_ts),
                'query_ts': qts,
                'query_time': unix_to_brisbane(qts),
                'distance_m': dist,
            
            }
            return best_match
    
    # Only return match if distance is within threshold

    return None
'''def find_nearest_gps_match(ref_lat, ref_lon, ref_ts, query_lat, query_lon, query_ts, window_r, window_q, match_dist):
    best_match = None
    best_dist = float('inf')
    for j, (qlat, qlon, qts) in enumerate(zip(query_lat, query_lon, query_ts)):
        dist = haversine_distance(ref_lat, ref_lon, qlat, qlon)
        if dist <= match_dist and dist < best_dist:
            best_dist = dist
            best_match = {
                'ref_lat': ref_lat,
                'ref_lon': ref_lon,
                'query_lat': qlat,
                'query_lon': qlon,
                'ref_ts': ref_ts,
                'ref_time': unix_to_brisbane(ref_ts),
                'query_ts': qts,
                'query_time': unix_to_brisbane(qts),
                'distance_m': dist,
            }
    return best_match'''

In [108]:
# Iterate through each query GPS point
# For each query point, find match in all 4 refs
# One row = query_ts, query_lat, query_lon, ref1_ts, ref1_lat, ref1_lon, ..., ref4_ts, ref4_lat, ref4_lon

rows = []
print(f"Iterating through {len(query_lat)} query points...")
print("-" * 60)

for q_idx in range(len(query_lat)):
    qlat = query_lat.iloc[q_idx]
    qlon = query_lon.iloc[q_idx]
    qts  = query_ts.iloc[q_idx]

    row = {
        'query_ts':  qts,
        'query_lat': qlat,
        'query_lon': qlon,
    }

    all_refs_found = True
    for i in range(len(folder_r_list)):
        match = find_nearest_gps_match(
            qlat, qlon, qts,                          # query point as "ref" arg
            ref_lat_list[i], ref_lon_list[i], ref_ts_list[i],  # ref as "query" arg
            window_q, window_r_list[i], MATCH_DIST
        )
        if match:
            row[f'ref{i+1}_ts']  = match['query_ts']   # matched point in ref
            row[f'ref{i+1}_lat'] = match['query_lat']
            row[f'ref{i+1}_lon'] = match['query_lon']
        else:
            row[f'ref{i+1}_ts']  = None
            row[f'ref{i+1}_lat'] = None
            row[f'ref{i+1}_lon'] = None
            all_refs_found = False

    rows.append(row)
    if q_idx % 100 == 0:
        print(f"  Query point {q_idx}/{len(query_lat)} done")

matches_df = pd.DataFrame(rows, columns=[
    'query_ts',  'query_lat',  'query_lon',
    'ref1_ts',   'ref1_lat',   'ref1_lon',
    'ref2_ts',   'ref2_lat',   'ref2_lon',
    'ref3_ts',   'ref3_lat',   'ref3_lon',
    'ref4_ts',   'ref4_lat',   'ref4_lon',
])

matches_df.to_csv(exp_path + f"/gps_matches_{exp_no}.csv", index=False)
print(f"\n✓ Saved matches_df with {len(matches_df)} rows and {len(matches_df.columns)} columns")
print(matches_df.head())

Iterating through 201 query points...
------------------------------------------------------------
  Query point 0/201 done
  Query point 100/201 done
  Query point 200/201 done

✓ Saved matches_df with 201 rows and 15 columns
     query_ts  query_lat   query_lon       ref1_ts   ref1_lat    ref1_lon  \
0  1587540272 -27.506722  152.911453  1.587453e+09 -27.506725  152.911476   
1  1587540273 -27.506742  152.911579  1.587453e+09 -27.506745  152.911589   
2  1587540274 -27.506762  152.911711  1.587453e+09 -27.506768  152.911713   
3  1587540275 -27.506783  152.911843  1.587453e+09 -27.506790  152.911844   
4  1587540276 -27.506805  152.911977  1.587453e+09 -27.506811  152.911974   

        ref2_ts   ref2_lat    ref2_lon       ref3_ts   ref3_lat    ref3_lon  \
0           NaN        NaN         NaN  1.587705e+09 -27.506748  152.911518   
1           NaN        NaN         NaN  1.587705e+09 -27.506748  152.911518   
2           NaN        NaN         NaN  1.587705e+09 -27.506787  152.9117

In [109]:
matches_df = pd.read_csv(r"C:\Arjun\Thesis\results\exp_4_all\e2.csv")
matches_df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Arjun\\Thesis\\results\\exp_4_all\\e2.csv'

In [ ]:
import json
exp_no = 4
config = {
    "experiment_name": f"dtw_test_{exp_no}",
    "data": {
        "query_folder": folder_q,
    },
    "max_ref_l": ref_length_max,
    "max_query_l": query_length_max,
    "pairs": []
}

pair_id = 1

for idx, row in matches_df.iloc[[0, 5]].iterrows():
    query_ts_val = row['query_ts']


    # Fix query window ONCE for this row (shared across all 4 refs)
    query_len = round(random.uniform(query_length_min, query_length_max), 1)
    query_offset_pct = random.uniform(0.2, 0.8)
    query_start = round(query_ts_val - (query_len * query_offset_pct), 1)
    query_end   = round(query_ts_val + (query_len * (1 - query_offset_pct)), 1)
    query_start = max(query_start, min(window_q))
    query_end   = min(query_end,   max(window_q))
    query_len   = query_end - query_start

    # One pair per ref
    for i in range(len(folder_r_list)):
        rfc = None
        if (i==0):
            rfc = "sunset1"
        elif(i==1):
            rfc = "night"
        elif(i==2):
            rfc = "daytime"
        else:
             rfc = "morning" 

        ref_ts_val = row[f'ref{i+1}_ts']

        # Skip if no match found for this ref
        if pd.isna(ref_ts_val):
            continue

        ref_len = round(random.uniform(ref_length_min, ref_length_max), 1)
        ref_offset_pct = random.uniform(0.2, 0.8)
        ref_start = round(ref_ts_val - (ref_len * ref_offset_pct), 1)
        ref_end   = round(ref_ts_val + (ref_len * (1 - ref_offset_pct)), 1)
        ref_start = max(ref_start, min(window_r_list[i]))
        ref_end   = min(ref_end,   max(window_r_list[i]))
        ref_len   = ref_end - ref_start

        pair = {
            "pair_id":     pair_id,
            "ref_id":      i + 1,
            "ref": rfc,
            "query_folder": folder_q,
            "ref_folder":  folder_r_list[i],
            "query_start": query_start,
            "query_end":   query_end,
            "ref_start":   ref_start,
            "ref_end":     ref_end,
            "ref_l":       ref_len,
            "query_l":     query_len,
            "ref_match":   ref_ts_val,
            "q_match":     query_ts_val,
            "gps_q":       [round(row['query_lat'], 6), round(row['query_lon'], 6)],
            "type":        "good"
        }
        config["pairs"].append(pair)
        pair_id += 1

with open(exp_path + f'/{exp_no}_mdtw_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f"✓ Generated config with {len(config['pairs'])} pairs")
print(f"  ({len(matches_df)} query rows × up to 4 refs = up to {len(matches_df)*4} pairs, minus missing matches)")

✓ Generated config with 8 pairs
  (6 query rows × up to 4 refs = up to 24 pairs, minus missing matches)
